# Chunking Benchmark trên Kaggle

Notebook này là lớp điều phối mỏng cho framework `chunkbench`: lấy code, chọn dữ liệu, kiểm tra adapter, chạy benchmark và xuất artifact. Không đặt implementation của chunker, metric hay adapter trong notebook.

Trước khi chạy: bật **Internet** trong Kaggle Settings. Notebook tự chuẩn hoá QASPER, HotpotQA `fullwiki/validation` (supporting-document subset) và UIT-ViQuAD từ Hugging Face. HotpotQA ở đây không phải retrieval trên toàn bộ Wikipedia; ViMQA vẫn cần dữ liệu được cấp quyền.

## 0. Cấu hình chung

Chỉ sửa cell này khi đổi tập dữ liệu, chế độ chạy hoặc commit cần tái lập. `smoke` dùng fixture có sẵn; `full` dùng dữ liệu thật đã được chuẩn bị ở bước 2. Mặc định chỉ chạy bốn baseline publishable; không trộn kết quả mock vào bảng chính.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ManhTanTran/kaggle-notebook.git"
REPO_REVISION = "main"  # Nên thay bằng commit SHA khi chốt kết quả nghiên cứu.
PROJECT_DIR = Path("/kaggle/working/chunking-benchmark")

RUN_MODE = "full"  # smoke | full
INSTALL_ADVANCED = True
REQUIRE_GPU = True
RESUME = True
FAIL_FAST = False
SEED = 42
MIN_MAPPING_RATE = 0.95

# QASPER, HotpotQA validation supporting-document subset và UIT-ViQuAD
# được notebook tải/chuẩn hoá trực tiếp từ Hugging Face. ViMQA vẫn cần
# dữ liệu adapter-ready được cấp quyền qua Kaggle Input.
SELECTED_DATASETS = ["qasper", "hotpotqa_fullwiki", "uit_viquad"]

SELECTED_METHODS = [
    "fixed_256",
    "fixed_512",
    "sentence_fixed_256",
    "sentence_fixed_512",
    "semantic_breakpoint",
    "semantic_single_linkage_paper_exact",
    "meta_ppl_raw",
    "meta_ppl_dynamic_512",
    "pic_paper_reimplementation",
    "pic_reimplementation_capped_512",
    "late_fixed_256",
    "late_fixed_512",
]

# Toàn bộ advanced method dùng real local models, không dùng mock backend.
REAL_EMBEDDING = {
    "name": "sentence_transformer",
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
}
REAL_METHOD_OVERRIDES = {
    "semantic_breakpoint": {
        "backend_type": "real",
        "threshold": {"type": "percentile", "value": 90},
        "min_chunk_tokens": 32,
    },
    "semantic_single_linkage_paper_exact": {
        "backend_type": "real",
        "target_clusters": 4,
        "lambda_weight": 0.5,
        "distance_threshold": 0.5,
    },
    "meta_ppl_raw": {
        "backend_type": "transformers",
        "model_name": "gpt2",
        "precision": "float16",
        "max_sequence_tokens": 512,
        "context_policy": "previous_segment",
        "prominence": 0.05,
    },
    "meta_ppl_dynamic_512": {
        "backend_type": "transformers",
        "model_name": "gpt2",
        "precision": "float16",
        "max_sequence_tokens": 512,
        "context_policy": "previous_segment",
        "prominence": 0.05,
        "max_chunk_tokens": 512,
    },
    "pic_paper_reimplementation": {
        "backend_type": "transformers",
        "summarizer_model_name": "google/flan-t5-small",
        "max_new_tokens": 64,
    },
    "pic_reimplementation_capped_512": {
        "backend_type": "transformers",
        "summarizer_model_name": "google/flan-t5-small",
        "max_new_tokens": 64,
        "max_chunk_tokens": 512,
    },
    "late_fixed_256": {
        "representation": {
            "backend_type": "transformers",
            "model_name": "sentence-transformers/all-MiniLM-L6-v2",
            "pooling": "mean",
            "normalize": True,
            "max_model_tokens": 512,
            "long_document_policy": "window",
            "window_stride": 64,
        }
    },
    "late_fixed_512": {
        "representation": {
            "backend_type": "transformers",
            "model_name": "sentence-transformers/all-MiniLM-L6-v2",
            "pooling": "mean",
            "normalize": True,
            "max_model_tokens": 512,
            "long_document_policy": "window",
            "window_stride": 64,
        }
    },
}

OUTPUT_ROOT = Path("/kaggle/working/benchmark_outputs")

DATA_ROOTS = {
    "qasper": PROJECT_DIR / "data" / "raw" / "qasper",
    "hotpotqa_fullwiki": PROJECT_DIR / "data" / "raw" / "hotpotqa_fullwiki",
    "uit_viquad": PROJECT_DIR / "data" / "raw" / "uit_viquad",
}

# Khi dùng ViMQA từ Kaggle Input, thêm nó vào DATA_ROOTS và
# SELECTED_DATASETS theo đúng layout mô tả trong docs/datasets/.


## 1. Clone, cài đặt và kiểm tra môi trường

In [ ]:
import importlib
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "fetch", "--tags", "origin"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", REPO_REVISION],
    check=True,
)
branch = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "symbolic-ref", "--quiet", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
if branch:
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", branch],
        check=True,
    )
os.chdir(PROJECT_DIR)

# Cài baseline hoặc toàn bộ real backends theo cấu hình ở Cell 0.
# Dùng đường dẫn tuyệt đối để kernel Kaggle luôn cài đúng project vừa clone.
package_spec = (
    f"{PROJECT_DIR}[dev,advanced]"
    if INSTALL_ADVANCED
    else f"{PROJECT_DIR}[dev]"
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-e",
        package_spec,
    ],
    check=True,
)

git_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()

# Kernel đang chạy không tự đọc lại .pth vừa tạo bởi editable install.
# Thêm src/ cho phiên hiện tại; subprocess ở các cell sau vẫn dùng package đã cài.
source_dir = str(PROJECT_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
chunkbench = importlib.import_module("chunkbench")

print("Project :", PROJECT_DIR)
print("Commit  :", git_commit)
print("Python  :", sys.version.split()[0])
print("Package :", chunkbench.__file__)

try:
    import torch
    print("PyTorch :", torch.__version__)
    print("CUDA    :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0))
    elif REQUIRE_GPU:
        raise RuntimeError(
            "Chọn GPU Accelerator trong Kaggle Settings trước khi chạy "
            "12 methods real."
        )
except ImportError:
    print("PyTorch chưa được cài; không cần cho baseline.")


## 2. Xác định/tải và xác thực dữ liệu

Notebook tự tải QASPER và UIT-ViQuAD 2.0 từ Hugging Face, rồi tạo local raw schema mà adapter cần trước khi validate.

- QASPER: thư mục chứa `validation.json` dạng paper-id keyed JSON.
- HotpotQA `fullwiki/validation`: notebook tạo `validation.json` và `corpus/articles.jsonl` từ Hugging Face. Corpus là union các context/supporting documents của validation, **không phải toàn bộ Wikipedia**.
- UIT-ViQuAD: thư mục chứa `validation.json` kiểu SQuAD.
- ViMQA: thư mục chứa `dev.json` hoặc `validation.json` đã được cấp quyền.

ViMQA cần dữ liệu đã được cấp quyền. HotpotQA được tải ở chế độ supporting-document subset để benchmark chunk/evidence retrieval khả thi trên Kaggle; không gọi kết quả này là FullWiki retrieval.

In [ ]:
import json as data_json

# QASPER trên Hub vẫn dùng dataset loading script, vì vậy cần datasets 3.x.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "datasets==3.6.0",
        "huggingface_hub>=0.23",
    ],
    check=True,
)

loaded_datasets = sys.modules.get("datasets")
if loaded_datasets is not None and loaded_datasets.__version__ != "3.6.0":
    raise RuntimeError(
        "Đã đổi phiên bản datasets. Hãy chọn Session > Restart Session, "
        "sau đó Run All từ Cell 0."
    )

datasets_module = importlib.import_module("datasets")
load_dataset = datasets_module.load_dataset
HfApi = importlib.import_module("huggingface_hub").HfApi

if RUN_MODE == "full" and "qasper" in SELECTED_DATASETS:
    qasper_id = "allenai/qasper"
    qasper_dir = DATA_ROOTS["qasper"]
    qasper_file = qasper_dir / "validation.json"

    if not qasper_file.exists():
        qasper_revision = HfApi().dataset_info(qasper_id).sha
        qasper_rows = load_dataset(
            qasper_id,
            split="validation",
            revision=qasper_revision,
            trust_remote_code=True,
        )
        qasper_papers = {}
        qasper_excluded_unobservable_questions = 0
        qasper_excluded_unobservable_evidence = 0

        for row in qasper_rows:
            sections = row["full_text"]
            converted_sections = [
                {"section_name": section_name, "paragraphs": paragraphs}
                for section_name, paragraphs in zip(
                    sections["section_name"], sections["paragraphs"], strict=True
                )
            ]
            float_captions = [
                f"FLOAT SELECTED: {caption}"
                for caption in (row.get("figures_and_tables") or {}).get("caption", [])
                if str(caption).strip()
            ]
            if float_captions:
                converted_sections.append(
                    {"section_name": "Figures and Tables", "paragraphs": float_captions}
                )
            searchable_text = "\n\n".join(
                [str(row["abstract"])]
                + [
                    str(paragraph)
                    for section in converted_sections
                    for paragraph in section["paragraphs"]
                ]
            )
            qas = row["qas"]
            converted_qas = []
            for index, question_id in enumerate(qas["question_id"]):
                # HF datasets 3.x/4.x có hai schema: answers là list hoặc dict-of-lists.
                raw_answers = qas["answers"]
                if isinstance(raw_answers, list):
                    annotations = raw_answers[index].get("answer", []) or []
                else:
                    annotations = raw_answers.get("answer", [])[index] or []
                supported_annotations = []
                for item in annotations:
                    answer = dict(item)
                    raw_evidence = answer.get("evidence") or []
                    supported_evidence = [
                        str(text)
                        for text in raw_evidence
                        if str(text) in searchable_text
                    ]
                    qasper_excluded_unobservable_evidence += (
                        len(raw_evidence) - len(supported_evidence)
                    )
                    if answer.get("unanswerable") or supported_evidence:
                        answer["evidence"] = supported_evidence
                        supported_annotations.append({"answer": answer})
                if supported_annotations:
                    converted_qas.append(
                        {
                            "question_id": str(question_id),
                            "question": str(qas["question"][index]),
                            "answers": supported_annotations,
                        }
                    )
                else:
                    qasper_excluded_unobservable_questions += 1

            qasper_papers[str(row["id"])] = {
                "title": row["title"],
                "abstract": [row["abstract"]] if row["abstract"] else [],
                "full_text": converted_sections,
                "qas": converted_qas,
            }

        qasper_dir.mkdir(parents=True, exist_ok=True)
        qasper_file.write_text(
            data_json.dumps(qasper_papers, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        (qasper_dir / "huggingface_source.json").write_text(
            data_json.dumps(
                {
                    "dataset_id": qasper_id,
                    "revision": qasper_revision,
                    "split": "validation",
                    "paper_count": len(qasper_papers),
                    "excluded_unobservable_question_count": (
                        qasper_excluded_unobservable_questions
                    ),
                    "excluded_unobservable_evidence_count": (
                        qasper_excluded_unobservable_evidence
                    ),
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    print("QASPER sẵn sàng:", qasper_file)

if RUN_MODE == "full" and "hotpotqa_fullwiki" in SELECTED_DATASETS:
    from chunkbench.data.normalization import normalize_title

    hotpot_id = "hotpotqa/hotpot_qa"
    hotpot_config = "fullwiki"
    hotpot_dir = DATA_ROOTS["hotpotqa_fullwiki"]
    hotpot_file = hotpot_dir / "validation.json"
    hotpot_corpus_dir = hotpot_dir / "corpus"
    hotpot_corpus_file = hotpot_corpus_dir / "articles.jsonl"
    hotpot_source_file = hotpot_dir / "huggingface_source.json"
    hotpot_revision = HfApi().dataset_info(hotpot_id).sha

    current_source = {
        "dataset_id": hotpot_id,
        "config": hotpot_config,
        "split": "validation",
        "revision": hotpot_revision,
        "corpus_mode": "global_supporting_document_subset",
        "not_full_wikipedia": True,
    }
    reusable = False
    if all(
        path.exists()
        for path in (hotpot_file, hotpot_corpus_file, hotpot_source_file)
    ):
        existing_source = data_json.loads(
            hotpot_source_file.read_text(encoding="utf-8")
        )
        reusable = all(
            existing_source.get(key) == value
            for key, value in current_source.items()
        )

    if not reusable:
        hotpot_rows = load_dataset(
            hotpot_id,
            hotpot_config,
            split="validation",
            revision=hotpot_revision,
        )
        questions = []
        articles = {}
        supporting_fact_count = 0

        for row in hotpot_rows:
            context = row["context"]
            titles = context["title"]
            sentence_groups = context["sentences"]
            facts = row["supporting_facts"]
            fact_titles = facts["title"]
            fact_indices = facts["sent_id"]
            if len(titles) != len(sentence_groups):
                raise ValueError(
                    f"HotpotQA context title/sentences mismatch: {row['id']}"
                )
            if len(fact_titles) != len(fact_indices) or not fact_titles:
                raise ValueError(f"HotpotQA supporting_facts invalid: {row['id']}")

            context_by_title = {}
            for title, sentences in zip(titles, sentence_groups, strict=True):
                canonical_title = str(title)
                normalized_title = normalize_title(canonical_title)
                normalized_sentences = [str(sentence) for sentence in sentences]
                prior = articles.get(normalized_title)
                if prior is not None and prior["text"] != [normalized_sentences]:
                    raise ValueError(
                        f"Conflicting HotpotQA article text: {canonical_title}"
                    )
                articles[normalized_title] = {
                    "title": canonical_title,
                    # Giữ một paragraph gồm các câu để sent_id chính thức còn đúng.
                    "text": [normalized_sentences],
                }
                context_by_title[normalized_title] = normalized_sentences

            supporting_facts = []
            for title, sentence_index in zip(fact_titles, fact_indices, strict=True):
                normalized_title = normalize_title(str(title))
                if normalized_title not in context_by_title:
                    raise ValueError(
                        "Supporting title absent from HF context: "
                        f"{row['id']} / {title}"
                    )
                sentence_index = int(sentence_index)
                if not 0 <= sentence_index < len(context_by_title[normalized_title]):
                    raise ValueError(
                        "Supporting sentence out of range: "
                        f"{row['id']} / {title} / {sentence_index}"
                    )
                supporting_facts.append([str(title), sentence_index])

            questions.append(
                {
                    "_id": str(row["id"]),
                    "question": str(row["question"]),
                    "answer": str(row["answer"]),
                    "supporting_facts": supporting_facts,
                }
            )
            supporting_fact_count += len(supporting_facts)

        hotpot_dir.mkdir(parents=True, exist_ok=True)
        hotpot_corpus_dir.mkdir(parents=True, exist_ok=True)
        hotpot_file.write_text(
            data_json.dumps(questions, ensure_ascii=False), encoding="utf-8"
        )
        with hotpot_corpus_file.open("w", encoding="utf-8") as handle:
            for article in articles.values():
                handle.write(data_json.dumps(article, ensure_ascii=False) + "\n")
        hotpot_source_file.write_text(
            data_json.dumps(
                {
                    **current_source,
                    "query_count": len(questions),
                    "article_count": len(articles),
                    "supporting_fact_count": supporting_fact_count,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    print(
        "HotpotQA supporting-document subset sẵn sàng:",
        hotpot_file,
        "(không phải FullWiki toàn Wikipedia)",
    )

if RUN_MODE == "full" and "uit_viquad" in SELECTED_DATASETS:
    uit_id = "taidng/UIT-ViQuAD2.0"
    uit_dir = DATA_ROOTS["uit_viquad"]
    uit_file = uit_dir / "validation.json"

    if not uit_file.exists():
        uit_revision = HfApi().dataset_info(uit_id).sha
        uit_rows = load_dataset(uit_id, split="validation", revision=uit_revision)
        articles = {}
        excluded_impossible = 0

        for row in uit_rows:
            raw_answers = row.get("answers") or {}
            texts = raw_answers.get("text") or []
            starts = raw_answers.get("answer_start") or []
            if row.get("is_impossible") or not texts:
                excluded_impossible += 1
                continue

            answers = [
                {"text": str(text), "answer_start": int(start)}
                for text, start in zip(texts, starts, strict=True)
                if str(text).strip()
            ]
            if not answers:
                excluded_impossible += 1
                continue

            title = str(row.get("title") or "")
            context = str(row["context"])
            article = articles.setdefault(
                title, {"title": title, "paragraphs": []}
            )
            paragraph = next(
                (item for item in article["paragraphs"] if item["context"] == context),
                None,
            )
            if paragraph is None:
                paragraph = {"context": context, "qas": []}
                article["paragraphs"].append(paragraph)
            paragraph["qas"].append(
                {
                    "id": str(row["id"]),
                    "question": str(row["question"]),
                    "answers": answers,
                }
            )

        squad = {"data": list(articles.values())}
        uit_dir.mkdir(parents=True, exist_ok=True)
        uit_file.write_text(
            data_json.dumps(squad, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        (uit_dir / "huggingface_source.json").write_text(
            data_json.dumps(
                {
                    "dataset_id": uit_id,
                    "revision": uit_revision,
                    "split": "validation",
                    "article_count": len(squad["data"]),
                    "excluded_impossible_count": excluded_impossible,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    print("UIT-ViQuAD sẵn sàng:", uit_file)

for name in SELECTED_DATASETS:
    if RUN_MODE == "full" and not DATA_ROOTS[name].exists():
        raise FileNotFoundError(f"Thiếu dữ liệu đã chuẩn bị: {DATA_ROOTS[name]}")
    print(f"{name}: {DATA_ROOTS[name]}")


In [ ]:
import yaml

from chunkbench.data.validation import validate_dataset
from chunkbench.registry.datasets import build_dataset_adapter

BASE_CONFIG = (
    Path("configs/experiments/all_qa_datasets_smoke.yaml")
    if RUN_MODE == "smoke"
    else Path("configs/experiments/all_qa_datasets_core_methods.yaml")
)
runtime_dir = PROJECT_DIR / "runtime_configs"
runtime_dir.mkdir(exist_ok=True)
experiment = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
dataset_items = experiment["datasets"]
runtime_dataset_configs = []
validation_reports = {}

for item in dataset_items:
    source_path = PROJECT_DIR / item
    dataset_config = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    dataset_name = dataset_config["name"]
    if dataset_name not in SELECTED_DATASETS:
        continue

    if RUN_MODE == "full":
        root = DATA_ROOTS[dataset_name]
        if dataset_name == "hotpotqa_fullwiki":
            dataset_config["questions_path"] = str(root / "validation.json")
            dataset_config["corpus_path"] = str(root / "corpus")
        else:
            dataset_config["data_path"] = str(root)

    adapter_name = dataset_config.get("adapter", dataset_name)
    bundle = build_dataset_adapter(adapter_name, dataset_config).load()
    report = validate_dataset(bundle, dataset_config.get("validation"))
    mapping_rate = report["evidence_mapping_rate"]
    if mapping_rate < MIN_MAPPING_RATE:
        raise RuntimeError(
            f"{dataset_name}: mapping rate {mapping_rate:.2%} < {MIN_MAPPING_RATE:.2%}"
        )

    target_path = runtime_dir / f"{dataset_name}.yaml"
    target_path.write_text(
        yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    runtime_dataset_configs.append(str(target_path))
    validation_reports[dataset_name] = report
    print(
        f"{dataset_name}: documents={report['document_count']}, "
        f"queries={report['query_count']}, evidence={report['evidence_count']}, "
        f"mapping={mapping_rate:.2%}"
    )

if not runtime_dataset_configs:
    raise RuntimeError("Không có dataset nào được chọn.")


## 3. Chạy smoke hoặc full benchmark

Cell này tạo một YAML runtime ở `/kaggle/working/chunking-benchmark/runtime_configs/`, không sửa config gốc trong repository. Nếu `RUN_MODE = full`, dữ liệu thật đã qua validation ở bước trước mới được chạy.

In [ ]:
methods = [
    {**spec, **REAL_METHOD_OVERRIDES.get(spec["name"], {})}
    for spec in experiment["methods"]
    if spec["name"] in SELECTED_METHODS
]
if len(methods) != len(SELECTED_METHODS):
    available = [spec["name"] for spec in experiment["methods"]]
    raise ValueError(f"Method không có trong profile hiện tại. Có: {available}")

# Không cho mock backend lọt vào một full benchmark có thể báo cáo.
if RUN_MODE == "full":
    mocked = [
        spec["name"] for spec in methods
        if spec.get("backend_type") == "mock"
        or spec.get("representation", {}).get("backend_type") == "mock"
    ]
    if mocked:
        raise ValueError(
            "Full benchmark không nhận mock methods: " + ", ".join(mocked)
        )

run_name = f"kaggle_{RUN_MODE}_{'_'.join(SELECTED_DATASETS)}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
experiment["run_name"] = run_name
experiment["datasets"] = runtime_dataset_configs
experiment["methods"] = methods
experiment["embedding"] = REAL_EMBEDDING
experiment["output_dir"] = str(OUTPUT_ROOT)
experiment["seed"] = SEED
experiment.setdefault("execution", {})
experiment["execution"].update({
    "resume": RESUME,
    "skip_completed": RESUME,
    "fail_fast": FAIL_FAST,
})

runtime_experiment = runtime_dir / "experiment.yaml"
runtime_experiment.write_text(
    yaml.safe_dump(experiment, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

command = [sys.executable, "-m", "chunkbench.cli", "--config", str(runtime_experiment)]
log_path = OUTPUT_ROOT / f"{run_name}.log"
print("Lệnh chạy:", " ".join(command))

process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
with log_path.open("w", encoding="utf-8") as log_file:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

if process.wait() != 0:
    raise RuntimeError(f"Benchmark thất bại; xem log: {log_path}")

RUN_DIR = OUTPUT_ROOT / run_name
required_artifacts = [
    RUN_DIR / "benchmark_metrics.csv",
    RUN_DIR / "chunk_statistics.csv",
    RUN_DIR / "experiment_manifest.json",
]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError("Thiếu artifact: " + ", ".join(missing_artifacts))
print(f"Hoàn tất: {RUN_DIR}")


## 4. Đọc, hiển thị và xuất kết quả

In [ ]:
import json

import pandas as pd

metrics_df = pd.read_csv(RUN_DIR / "benchmark_metrics.csv")
chunk_stats_df = pd.read_csv(RUN_DIR / "chunk_statistics.csv")
failed_path = RUN_DIR / "failed_runs.json"
failed_runs = (
    json.loads(failed_path.read_text(encoding="utf-8"))
    if failed_path.exists()
    else []
)

publishable_df = metrics_df.copy()
if "IsMockBackend" in publishable_df:
    publishable_df = publishable_df[
        ~publishable_df["IsMockBackend"].fillna(False)
    ]
if "IsPublishableBenchmark" in publishable_df:
    publishable_df = publishable_df[
        publishable_df["IsPublishableBenchmark"].fillna(False)
    ]

columns = [
    "Dataset", "Method", "Hit@10", "MRR@10",
    "EvidenceRecallMacro@10", "EvidenceCoverage@10",
    "EvidenceRecallMacro@2048Tokens", "EvidenceCoverage@2048Tokens",
    "Redundancy@10", "Tokens per Chunk Mean", "RuntimeSeconds",
]
columns = [column for column in columns if column in publishable_df.columns]

print(
    f"Metric rows: {len(metrics_df)} | "
    f"Chunk-stat rows: {len(chunk_stats_df)} | Failed runs: {len(failed_runs)}"
)
display(
    publishable_df[columns].sort_values(
        ["Dataset", "EvidenceRecallMacro@10"], ascending=[True, False]
    )
)

if failed_runs:
    display(pd.DataFrame(failed_runs))


In [ ]:
import shutil

# Mỗi dataset có một bảng riêng với toàn bộ 23 primary metrics.
from chunkbench.eval.constants import PRIMARY_METRICS

metric_labels = {
    "EvidenceRecallMacro@3": "Ev. Recall Macro@3",
    "EvidenceRecallMacro@5": "Ev. Recall Macro@5",
    "EvidenceRecallMacro@10": "Ev. Recall Macro@10",
    "EvidenceRecallMicro@3": "Ev. Recall Micro@3",
    "EvidenceRecallMicro@5": "Ev. Recall Micro@5",
    "EvidenceRecallMicro@10": "Ev. Recall Micro@10",
    "DocumentRecall@3": "Doc. Recall@3",
    "DocumentRecall@5": "Doc. Recall@5",
    "DocumentRecall@10": "Doc. Recall@10",
    "EvidenceCoverage@3": "Ev. Coverage@3",
    "EvidenceCoverage@5": "Ev. Coverage@5",
    "EvidenceCoverage@10": "Ev. Coverage@10",
    "EvidenceRecallMacro@2048Tokens": "Ev. Recall Macro@2048T",
    "EvidenceCoverage@2048Tokens": "Ev. Coverage@2048T",
}
report_columns = ["Method", *PRIMARY_METRICS]
available_columns = [
    column for column in report_columns if column in publishable_df.columns
]

for dataset_name, dataset_metrics in publishable_df.groupby("Dataset", sort=True):
    print(f"\n## Kết quả: {dataset_name}")
    report_table = dataset_metrics[available_columns].rename(columns=metric_labels)
    report_table = report_table.sort_values(
        "Ev. Recall Macro@10", ascending=False
    )
    percent_columns = [column for column in report_table if column != "Method"]
    display(
        report_table.style.format({column: "{:.2%}" for column in percent_columns})
        .set_properties(**{"text-align": "right", "white-space": "nowrap"})
        .set_properties(subset=["Method"], **{"text-align": "left"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "table", "props": [("font-size", "11px")]},
        ])
    )
    report_path = RUN_DIR / f"{dataset_name}_full_metrics.csv"
    report_table.to_csv(report_path, index=False)
    print(f"Đã xuất đủ 23 metric: {report_path.name}")
    dataset_failed = [
        item for item in failed_runs if item.get("dataset") == dataset_name
    ]
    if dataset_failed:
        failed_methods = ", ".join(item["method"] for item in dataset_failed)
        print("Chưa có metric cho:", failed_methods)

archive_path = shutil.make_archive(
    base_name=str(Path("/kaggle/working") / f"{run_name}_results"),
    format="zip",
    root_dir=RUN_DIR,
)
print("Đã tạo file tải về từ Kaggle Output:", archive_path)
